# Appendix 2 — Validating the Production Model against the Benchmark

The full corpus is coded by Qwen3.5-122B-A10B (4-bit quantized), an open-weights model
served locally with [vLLM](https://docs.vllm.ai) behind a standard OpenAI-compatible API — the
code below works against any OpenAI-compatible endpoint. Each decision is coded in a single
call that asks all of the study's questions at once, at temperature 0.

This notebook contains, in order:

1. the complete prompt sent to the model (verbatim);
2. the run code that coded the 600 cases of the benchmark sample (guarded by
   `RUN_MODEL = False`; the study's codes are committed as `data/golden_predictions_qwen.json`,
   so the evaluation reproduces without re-running the model);
3. the evaluation of the model's codes against the reference labels.

> **About the reference labels.** The benchmark sample (Appendix 1) was coded by Claude Fable 5 — machine coding, not human ground truth. Wherever this appendix says “accuracy,” read “agreement with those reference labels.” The human check on the pipeline is the reasonableness review of Appendix 4.

The four coded questions (`timing`, `outcome`, `basis`, and the Federal-Court-only `jr_refugee` gate) and their allowed codes are set out in Appendix 1, with the full rubric in `data/golden_sample_coding_rubric.md`; the exact wording the model sees is the prompt below.

AI Disclosure: Generative AI tools were used in preparing these appendices, including for background research, data extraction, coding, drafting and editing (Claude Code, using Fable 5 and 5.5 and Opus 5 and 5.5).

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

DATA = Path("data")          # every file these appendices need ships in this data/ folder
OUT = DATA / "qwen_vllm"     # one JSON per benchmark case - created only if you re-run the model below

## The prompt

Everything the model sees is below: a short system message, a rubric per question (the timing
rubric in particular spells out which inferences are expected), and the decision text. The model
must end its reply with a single line of JSON. Tribunal cases (RPD/RAD/RLLR) get three
questions; Federal Court cases get the `jr_refugee` gate plus the three questions, with
instructions to answer `null` when gated out.

In [2]:
KIND = {
    "RPD": "Refugee Protection Division (RPD) first-instance refugee determination",
    "RAD": "Refugee Appeal Division (RAD) decision on an appeal from the RPD",
    "RLLR": "published Refugee Protection Division (RPD) refugee determination",
    "FC": "Federal Court judicial review decision",
}

SYSTEM = (
    "You are an expert legal research assistant coding Canadian refugee law decisions "
    "for an empirical study. You follow the coding rubric exactly, base every answer only "
    "on the text of the decision (or reasonable inference from it), and answer in the "
    "required format with no extra commentary."
)

TIMING_RUBRIC = """Was the refugee claim made MORE THAN ONE YEAR after the principal claimant's FIRST entry to Canada?

(a) CLEAR (on the text or by reasonable inference) that the claim was made MORE than one year after the principal claimant's FIRST entry to Canada.
(b) CLEAR that the claim was made ONE YEAR OR LESS after the principal claimant's first entry to Canada.
(c) Unclear.

Rules:
- "First entry" means the FIRST time the principal claimant ever entered Canada in any status (visitor, student, worker, an earlier visit years before), NOT the most recent entry. If the claimant had an earlier entry to Canada more than a year before the claim (e.g. visits in 2007 before a 2010 claim), answer (a) even if the claim was made promptly after the LATEST arrival.
- If the decision gives an arrival date and a claim/referral date, compute the gap.
- REASONABLE INFERENCE IS EXPECTED, not just allowed. Do NOT answer (c) merely because no explicit claim date is stated. Answer (b) whenever the narrative indicates the claim followed the first arrival within a year, including any of these patterns:
  * the claim was made at the port of entry, at the border, or "on arrival"/"upon arrival";
  * the claimant fled their country and travelled to Canada in order to seek protection (e.g. "fled to Canada", "came to Canada and claimed refugee protection"), with no suggestion of delay in claiming and no earlier entry to Canada;
  * the claim was made, referred, or the Basis of Claim signed in the same year as the first arrival, or the arrival and the claim are recounted as one continuous sequence of events;
  * the RPD/RAD heard or decided the claim within about a year of the first arrival (the claim was necessarily made before the hearing/decision);
  * the panel discusses a delay in claiming of less than one year (e.g. "a nine-month delay in claiming").
- Answer (c) only when the text genuinely cannot resolve the one-year threshold either way: no arrival date AND no claim-timing narrative; or key dates redacted (e.g. "arrived XXXX 2022", claim "XXXX 2023") such that the gap could be under or over one year; or only vague statements ("some time after arriving") with nothing else to go on.
- For appeal or judicial review decisions, the question is still about when the underlying refugee claim was made relative to the principal claimant's first entry, using the facts recited in the decision."""

OUTCOME_TRIBUNAL_RUBRIC = """Was the PRINCIPAL claimant clearly successful or unsuccessful in this decision?

(a) Successful: obtained refugee protection (Convention refugee or person in need of protection); or, at the RAD, the appeal was ALLOWED — including where the RAD substitutes protection AND where the RAD refers the matter back to the RPD for redetermination (both count as success).
(b) Unsuccessful: refugee protection denied; or, at the RAD, appeal dismissed / RPD decision confirmed against the claimant; or the claim was declared abandoned or vacated. For a MINISTER's appeal at the RAD, code from the claimant's perspective: Minister wins -> (b); Minister's appeal dismissed (claimant keeps protection) -> (a).
(c) Unclear (e.g. interlocutory or purely procedural ruling with no disposition of the claim, outcome not stated, or a split/mixed outcome for the principal claimant that cannot be resolved).

Rules:
- Code only the PRINCIPAL claimant. If the principal claimant succeeds but associated claimants fail, answer (a); the reverse, (b)."""

OUTCOME_FC_RUBRIC = """This is a judicial review of a refugee determination. What was the outcome of the JUDICIAL REVIEW, coded from the REFUGEE CLAIMANT's perspective?

(a) The claimant succeeds: the application for judicial review is granted/allowed (the RPD/RAD decision is set aside and the matter returned for redetermination) where the claimant is the applicant; OR, where the MINISTER is the applicant, the Minister's application is dismissed (the claimant keeps the favourable determination).
(b) The claimant is unsuccessful: the claimant's application for judicial review is dismissed; OR the Minister's application against a favourable determination is granted.
(c) Unclear (e.g. outcome not stated, interlocutory ruling, costs or certified-question ruling with no disposition of the judicial review itself)."""

BASIS_RUBRIC = """Did the claim clearly involve ALLEGATIONS of persecution based on the following? Code what was ALLEGED, not what the tribunal or court found credible or established. Consider the principal claimant AND any dependants/associated claimants in the same decision.

(a) Gender-based persecution — e.g. domestic/intimate-partner violence, forced marriage, FGM, honour-based violence, rape or other sexual violence, forced abortion/sterilization or other reproductive coercion, gender-specific dress codes or social norms, persecution of women as a particular social group — and NOT sexual-orientation-based persecution.
(b) Sexual orientation, gender identity, or gender expression (SOGIE) based persecution — and NOT gender-based persecution.
(c) BOTH (a) and (b) are present (intersecting allegations, possibly split across the principal claimant and dependants).
(d) Another form of persecution ONLY (e.g. political opinion, religion, ethnicity or race, caste, criminal gangs or cartels, blood feuds, witness retaliation), where none of (a)/(b)/(c) applies.
(e) Unclear what type of persecution was alleged (e.g. a purely procedural decision that never describes the claim).

Rules:
- A claim by a woman is not automatically gender-based: the ALLEGED harm must itself be gender-based, not purely political/ethnic/criminal.
- If gender-based harm is alleged for any claimant or dependant alongside other grounds, that is (a) (or (c) if SOGIE allegations are also present); (d) applies only when NO gender or SOGIE allegation is present."""

JR_RUBRIC = """This is a Federal Court decision. Is it a judicial review of a refugee determination made by the Refugee Protection Division (RPD) and/or the Refugee Appeal Division (RAD) of the Immigration and Refugee Board?

(a) Yes: the decision under review is an RPD or RAD refugee determination — including RPD/RAD decisions on the merits, on vacation or cessation of refugee protection, and RPD/RAD refusals to reopen or reinstate a refugee claim or appeal.
(b) No: the decision under review is something else (e.g. a Pre-Removal Risk Assessment (PRRA), humanitarian and compassionate (H&C) application, visa or permanent-residence refusal, detention review, removal or deferral decision, inadmissibility finding, citizenship matter, sponsorship appeal, or a non-immigration matter) — even if the applicant's refugee history with the RPD/RAD is described as background.
(c) Unknown: cannot tell from the text what decision is under review."""

COMBINED_TRIBUNAL = """You will read a Canadian {kind}, then answer THREE coding questions.

QUESTION 1 (timing):

{timing}

QUESTION 2 (outcome):

{outcome}

QUESTION 3 (basis):

{basis}

THE DECISION TEXT:

<<<DECISION START>>>
{text}
<<<DECISION END>>>

Now answer all three questions about this decision. Your reply must END with a single line of JSON in exactly this form (lowercase letters only):

{{"timing": "<a|b|c>", "outcome": "<a|b|c>", "basis": "<a|b|c|d|e>"}}"""

COMBINED_FC = """You will read a Canadian {kind}, then answer coding questions.

QUESTION 0 (gate):

{jr}

If your answer to QUESTION 0 is (b) or (c), answer null for all remaining questions. Only if it is (a), answer them as follows.

QUESTION 1 (timing):

{timing}

QUESTION 2 (outcome):

{outcome}

QUESTION 3 (basis):

{basis}

THE DECISION TEXT:

<<<DECISION START>>>
{text}
<<<DECISION END>>>

Now answer about this decision. Your reply must END with a single line of JSON in exactly this form (lowercase letters, or null where gated):

{{"jr_refugee": "<a|b|c>", "timing": "<a|b|c or null>", "outcome": "<a|b|c or null>", "basis": "<a|b|c|d|e or null>"}}"""


def combined_messages(dataset, text):
    """Assemble the two-message conversation (system + user) for one decision."""
    kind = KIND[dataset]
    if dataset == "FC":
        user = COMBINED_FC.format(kind=kind, jr=JR_RUBRIC, timing=TIMING_RUBRIC,
                                  outcome=OUTCOME_FC_RUBRIC, basis=BASIS_RUBRIC, text=text)
    else:
        user = COMBINED_TRIBUNAL.format(kind=kind, timing=TIMING_RUBRIC,
                                        outcome=OUTCOME_TRIBUNAL_RUBRIC, basis=BASIS_RUBRIC, text=text)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


JSON_RE = re.compile(r"\{[^{}]*\}")

def parse_json(content):
    """Return the last {...} JSON object in the model's reply (the answer line)."""
    out = {}
    for m in JSON_RE.findall(content or ""):
        try:
            d = json.loads(m)
            if isinstance(d, dict):
                out = d
        except json.JSONDecodeError:
            pass
    return out


## Running the model on the 600 cases

The next cell is the complete run loop. It is off by default (`RUN_MODEL = False`): the
study's codes are committed in `data/golden_predictions_qwen.json` and the evaluation below
reads that file. To re-run, write the decision texts with `WRITE_CASE_FILES = True` in
Appendix 1, point `VLLM_QWEN35_BASE_URL` at an OpenAI-compatible server hosting the model (export
it in the shell or put it in a `.env` file in the repository root, which the run cell reads), and
set `RUN_MODEL = True`; a case whose output file already exists is skipped, so an interrupted
run resumes where it left off.

The study's actual run used this exact prompt and configuration, run asynchronously with
several requests in flight for speed. All 600 cases finished cleanly: `finish_reason` “stop” on
the first rung of the retry ladder, and every reply ended with parseable JSON.

In [3]:
gold = pd.read_json(DATA / "golden_ai_generated_sample.json")
print(f"{len(gold)} cases in the benchmark sample")

600 cases in the benchmark sample


In [4]:
RUN_MODEL = False   # set True (with a model server running) to re-code the benchmark cases yourself

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # reads VLLM_QWEN35_BASE_URL from a .env file in the repository root, if there is one

BASE_URL = os.environ.get("VLLM_QWEN35_BASE_URL", "http://localhost:8000/v1")
MODEL = "qwen35-122b"

def code_one_case(client, dataset, text):
    """Send one decision to the model and return its codes plus bookkeeping fields.

    The model is allowed to "think" before answering. In rare cases (well under 1%)
    the thinking runs away and exhausts the output budget, so we walk a retry
    ladder: normal budget -> doubled budget -> thinking disabled. We always check
    finish_reason: anything other than "stop" would mean a truncated (unreliable)
    answer. temperature=0 makes runs as repeatable as the server allows.
    """
    for max_tokens, think in [(32768, True), (65536, True), (32768, False)]:
        extra = {} if think else {"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}
        resp = client.chat.completions.create(
            model=MODEL, messages=combined_messages(dataset, text),
            temperature=0, max_tokens=max_tokens, timeout=1800, **extra)
        choice = resp.choices[0]
        if choice.finish_reason != "length":
            break
    return {"parsed": parse_json(choice.message.content),
            "finish_reason": choice.finish_reason,
            "input_tokens": resp.usage.prompt_tokens,
            "output_tokens": resp.usage.completion_tokens,
            "thinking_disabled": not think}

if RUN_MODEL:
    client = OpenAI(base_url=BASE_URL, api_key="not-needed")
    OUT.mkdir(exist_ok=True)
    for _, g in gold.iterrows():
        path = OUT / f"{g.case_id}.json"
        if path.exists():                     # resumable: already-coded cases are skipped
            continue
        # decision texts: written by Appendix 1 with WRITE_CASE_FILES = True
        text = (DATA / "cases" / f"{g.case_id}.txt").read_text(encoding="utf-8")
        rec = {"case_id": g.case_id, **code_one_case(client, g.dataset, text)}
        path.write_text(json.dumps(rec))
        print(g.case_id, rec["parsed"], rec["finish_reason"])
else:
    print("RUN_MODEL is False - the evaluation below uses the committed predictions of the study's run")

RUN_MODEL is False - the evaluation below uses the committed predictions of the study's run


## Evaluation: agreement with the reference labels

The evaluation is end to end: for a Federal Court case that the model wrongly gates out, its
missing timing/outcome/basis answers count as wrong. The gate is scored over the 200 Federal
Court cases; the three main questions are scored over the 555 benchmark-gated-in cases (the 400
tribunal cases plus the 155 Federal Court cases the reference labels gate in). Because the gate
agreed 200/200, no case was scored wrong for being wrongly gated out, so every disagreement
below is a disagreement on the question itself. Each row of the table gives the number of cases
agreed, the number scored, and the share.

In [5]:
PRED_FILE = DATA / "golden_predictions_qwen.json"   # the study's model codes, one record per case

if OUT.exists():   # you re-ran the model above: evaluate your own outputs
    rows = []
    for _, g in gold.iterrows():
        path = OUT / f"{g.case_id}.json"
        if not path.exists():
            continue
        codes = json.loads(path.read_text()).get("parsed", {})
        rows.append({
            "case_id": g.case_id, "dataset": g.dataset,
            "gold_jr": g.jr_refugee if isinstance(g.jr_refugee, str) else None,
            "pred_jr": codes.get("jr_refugee"),
            "gold_timing": g.timing if isinstance(g.timing, str) else None,
            "pred_timing": codes.get("timing"),
            "gold_outcome": g.outcome if isinstance(g.outcome, str) else None,
            "pred_outcome": codes.get("outcome"),
            "gold_basis": g.basis if isinstance(g.basis, str) else None,
            "pred_basis": codes.get("basis"),
        })
    ev = pd.DataFrame(rows)
    print(f"{len(ev)} of {len(gold)} cases have model output")
else:              # default: the committed predictions from the study's run
    ev = pd.read_json(PRED_FILE)
    print(f"loaded {len(ev)} saved predictions from {PRED_FILE.name}")

def accuracy(d):
    """Per question: cases agreed, cases scored (those with a reference label), and the share."""
    rows = {}
    fc = d[d.gold_jr.notna()]
    rows["FC gate (jr_refugee)"] = ((fc.pred_jr == fc.gold_jr).sum(), len(fc))
    for q in ["timing", "outcome", "basis"]:
        s = d[d[f"gold_{q}"].notna()]
        rows[q] = ((s[f"pred_{q}"] == s[f"gold_{q}"]).sum(), len(s))
    out = pd.DataFrame(rows, index=["agreed", "scored"]).T
    out["agreement with the reference labels"] = (out.agreed / out.scored).round(3)
    return out

accuracy(ev)

loaded 600 saved predictions from golden_predictions_qwen.json


,agreed,scored,agreement with the reference labels
FC gate (jr_refugee),200,200,1.000
timing,503,555,0.906
outcome,554,555,0.998
basis,535,555,0.964


### Where the disagreements are

Confusion matrices (rows = reference label, columns = model). The pattern of the errors matters
more than their count: for the study, the worrying direction would be the model *inventing* late
claims or gender/SOGIE allegations that are not there.

In [6]:
def confusion(q):
    s = ev[ev[f"gold_{q}"].notna()]
    return pd.crosstab(s[f"gold_{q}"].rename("reference label"),
                       s[f"pred_{q}"].fillna("(none)").rename("model prediction"))

print("timing  (a = claim >1 year after first entry, b = within 1 year, c = unclear)")
display(confusion("timing"))
print("basis  (a = gender, b = SOGIE, c = both, d = other, e = unclear)")
display(confusion("basis"))
print("outcome  (a = successful, b = unsuccessful, c = unclear)")
display(confusion("outcome"))

timing  (a = claim >1 year after first entry, b = within 1 year, c = unclear)


model prediction,a,b,c
reference label,,,
a,37,2,7
b,2,316,19
c,0,22,150


basis  (a = gender, b = SOGIE, c = both, d = other, e = unclear)


model prediction,a,b,c,d,e
reference label,,,,,
a,105,0,0,2,0
b,0,62,2,0,0
c,1,2,8,0,0
d,12,0,0,341,0
e,0,0,0,1,19


outcome  (a = successful, b = unsuccessful, c = unclear)


model prediction,a,b,c
reference label,,,
a,238,0,0
b,1,312,0
c,0,0,4


### The paper's Table 2

The agreement figures above, assembled row for row as the paper's Table 2 reports them
(share agreed, with cases agreed of cases scored): per-question agreement, then the two
detection rows — late claims, and any gender/SOGIE allegation (basis a, b or c) — as precision
(how often the benchmark agrees when the model asserts the category) and recall (how many of
the benchmark's cases the model also finds).


In [7]:
def cell(k, n):
    return f"{100 * k / n:.1f}% ({k} of {n})"

acc = accuracy(ev)
table2 = [(name, cell(acc.loc[key, "agreed"], acc.loc[key, "scored"]))
          for key, name in [("FC gate (jr_refugee)", "Federal Court screen"), ("timing", "Timing"),
                            ("outcome", "Outcome"), ("basis", "Basis")]]

s = ev[ev.gold_timing.notna()]
g, p = s.gold_timing == "a", s.pred_timing == "a"
table2 += [("Late claims: when the model said late, the benchmark agreed", cell((g & p).sum(), p.sum())),
           ("Late claims: of the benchmark's late claims, the model also found", cell((g & p).sum(), g.sum()))]
s = ev[ev.gold_basis.notna()]
g, p = s.gold_basis.isin(["a", "b", "c"]), s.pred_basis.isin(["a", "b", "c"])
table2 += [("Any gender/SOGIE allegation: overall agreement", cell((g == p).sum(), len(s))),
           ("Any gender/SOGIE allegation: when the model found one, the benchmark agreed", cell((g & p).sum(), p.sum())),
           ("Any gender/SOGIE allegation: of the benchmark's, the model also found", cell((g & p).sum(), g.sum()))]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(table2, columns=["question", "agreement with the benchmark"]).set_index("question"))


,agreement with the benchmark
question,
Federal Court screen,100.0% (200 of 200)
Timing,90.6% (503 of 555)
Outcome,99.8% (554 of 555)
Basis,96.4% (535 of 555)
"Late claims: when the model said late, the benchmark agreed",94.9% (37 of 39)
"Late claims: of the benchmark's late claims, the model also found",80.4% (37 of 46)
Any gender/SOGIE allegation: overall agreement,97.5% (541 of 555)
"Any gender/SOGIE allegation: when the model found one, the benchmark agreed",93.8% (180 of 192)
"Any gender/SOGIE allegation: of the benchmark's, the model also found",98.9% (180 of 182)


### The disagreements, broken down

Two further views of the same 555 gated-in cases: the 52 timing disagreements partitioned by
whether a late code is involved on either side, and the direction of the basis disagreements on
the binary that matters for the study (any gender/SOGIE allegation vs none).

In [8]:
scored = ev[ev.gold_timing.notna()]
print(f"cases scored on the three main questions: {(scored.dataset != 'FC').sum()} tribunal + "
      f"{(scored.dataset == 'FC').sum()} gated-in FC = {len(scored)}\n")

# the timing disagreements, partitioned by whether a late code (a) is involved
s = ev[ev.gold_timing.notna()]
dis = s[s.gold_timing != s.pred_timing]
missed = dis[dis.gold_timing == "a"]                                   # reference late, model not
false_late = dis[(dis.gold_timing != "a") & (dis.pred_timing == "a")]  # model late, reference not
boundary = dis[(dis.gold_timing != "a") & (dis.pred_timing != "a")]    # no late code on either side
assert len(missed) + len(false_late) + len(boundary) == len(dis)
print(f"timing disagreements: {len(dis)} of {len(s)} scored cases "
      f"({len(missed) + len(false_late)} involve a late code on one side, {len(boundary)} do not)")
print(f"  missed late claims (reference a, model not a): {len(missed)}"
      f"  -> model said unclear: {(missed.pred_timing == 'c').sum()}, timely: {(missed.pred_timing == 'b').sum()}")
print(f"  false late claims (model a, reference not a):  {len(false_late)}"
      f"  -> reference was timely: {(false_late.gold_timing == 'b').sum()}, unclear: {(false_late.gold_timing == 'c').sum()}")
print(f"  timely/unclear boundary calls (no late code on either side): {len(boundary)}"
      f"  -> reference timely, model unclear: {((boundary.gold_timing == 'b') & (boundary.pred_timing == 'c')).sum()};"
      f" reference unclear, model timely: {((boundary.gold_timing == 'c') & (boundary.pred_timing == 'b')).sum()}")

cases scored on the three main questions: 400 tribunal + 155 gated-in FC = 555

timing disagreements: 52 of 555 scored cases (11 involve a late code on one side, 41 do not)
  missed late claims (reference a, model not a): 9  -> model said unclear: 7, timely: 2
  false late claims (model a, reference not a):  2  -> reference was timely: 2, unclear: 0
  timely/unclear boundary calls (no late code on either side): 41  -> reference timely, model unclear: 19; reference unclear, model timely: 22


In [9]:
# direction of the basis disagreements on the binary (any gender/SOGIE allegation vs none)
s = ev[ev.gold_basis.notna()].copy()
s["ref_gs"] = s.gold_basis.isin(["a", "b", "c"])
s["model_gs"] = s.pred_basis.isin(["a", "b", "c"])
over = s[~s.ref_gs & s.model_gs]    # reference: no gender/SOGIE allegation; model: gender/SOGIE
under = s[s.ref_gs & ~s.model_gs]   # the reverse
print(f"cases with any gender/SOGIE allegation: model {s.model_gs.sum()} vs reference {s.ref_gs.sum()} (of {len(s)})")
print(f"  reference none -> model gender/SOGIE (over-assignment):  {len(over)}")
print(f"  reference gender/SOGIE -> model none (under-assignment): {len(under)}")


cases with any gender/SOGIE allegation: model 192 vs reference 182 (of 555)
  reference none -> model gender/SOGIE (over-assignment):  12
  reference gender/SOGIE -> model none (under-assignment): 2


### The late share on the benchmark sample, two ways

The share of clear-timing decisions coded late, on the same 555 cases, according to the
benchmark labels and according to the model. Appendix 5 reports the model's late share on the
full corpus; the comparison below is the basis for reading that figure as a probable
under-estimate.

In [10]:
s = ev[ev.gold_timing.notna()]
ref_clear = s[s.gold_timing.isin(["a", "b"])]
model_clear = s[s.pred_timing.isin(["a", "b"])]
print(f"late share among clear-timing decisions, benchmark labels: "
      f"{(ref_clear.gold_timing == 'a').sum()}/{len(ref_clear)} = {100 * (ref_clear.gold_timing == 'a').mean():.1f}%")
print(f"late share among clear-timing decisions, model codes:      "
      f"{(model_clear.pred_timing == 'a').sum()}/{len(model_clear)} = {100 * (model_clear.pred_timing == 'a').mean():.1f}%")

late share among clear-timing decisions, benchmark labels: 46/383 = 12.0%
late share among clear-timing decisions, model codes:      39/379 = 10.3%


### Reading the results

- The Federal Court gate and the outcome question are essentially solved (gate 200/200; one
  outcome disagreement in 555 gated-in cases).
- Timing is the hard question (503/555 ≈ 0.91 agreement over the gated-in cases). Its errors
  lean conservative for the study's key group: the model codes a claim as late against a timely
  reference label in only 2 cases, and never resolves an “unclear” reference to late. For claims
  made more than one year after first entry, precision is ≈0.95 and recall ≈0.80: the late group
  is under-counted, not inflated. Of the 52 disagreements, 9 are missed late claims (7 coded
  “unclear”, 2 “timely”), 2 are false late claims, and the remaining 41 — every disagreement that
  does not involve a late code on either side — sit on the soft boundary between “timely” and
  “unclear” (22 reference-“unclear” cases coded “timely”, 19 the reverse), the softest calls in
  the scheme (arrival narrative present, claim date absent).
- On the benchmark sample itself, the late share among clear-timing decisions is 46/383 = 12.0%
  by the reference labels and 39/379 = 10.3% by the model on the same cases: the model's late
  share runs below the reference's, so the corpus-level late share in Appendix 5 is more likely
  an under-estimate than an over-estimate.
- Basis errors sit almost entirely on the contested boundary between “gender-based
  persecution was alleged” and “the harm occurred inside a claim alleged on other grounds” — the
  same cases the reference coding flagged as judgment calls. On the binary that matters for the
  study (any gender/SOGIE allegation vs none), agreement is 541/555 ≈ 0.975. Unlike timing, the
  direction here is an over-count: relative to the benchmark the model slightly *over*-counts
  gender-based allegations (192 cases with any gender/SOGIE allegation against the reference's
  182; 12 over-assignments against 2 under-assignments).

This model and configuration — validated here against the benchmark sample — were
adopted for the full-corpus production run (Appendix 3).

### A closer look at the individual disagreements

The disagreements were also reviewed case by case by Claude Fable 5 (running in Claude Code) —
the reference coder reviewing its own disagreements, an obvious caveat — reading each divergent
code alongside the reference evidence sentence and, where needed, the decision text. The
divergences cluster tightly:

- **Every timing disagreement that does not involve a late code sits on the “timely” /
  “unclear” boundary (41 of 52; the other 11 are the 9 missed and 2 false late claims).** The
  recurring shape: an arrival date and a fled-to-Canada narrative but no explicit claim date,
  often with a redacted month. The rubric's inference patterns (“fled to Canada and claimed
  protection,” one continuous sequence of events) invite the ≤ 1-year inference; whichever coder
  declines it answers “unclear.” Many of these calls are defensible either way.
- **The missed late claims (9) need an extra inferential step the model declines.** Several turn
  on dating the claim from its IRB file-number prefix (e.g. reading a `TC1-` file as a 2021
  Toronto referral) — a convention the reference coding uses and the production model does not;
  the rest require assembling lateness from indirect facts (a first entry years before a return
  home and re-entry; a principal claimant who arrived years before the dependants whose arrival
  prompted the claim). The model answers “unclear” (7 cases) or “timely” (2) instead.
- **The late-claim group stays clean.** Only 2 of the model's 39 late codes disagree with the
  reference label. Errors on the study's key group are omissions, not contamination: the
  over-one-year group is under-counted, not inflated.
- **Outcome: the one error is a slip, not an edge case.** The decision in `304_RAD` states
  “I dismiss the appeal and confirm the decision of the RPD”; the model answered “successful.”
  One such error in 555 gated-in cases, with no pattern behind it.
- **Basis: none of the errors invents gender/SOGIE content from nothing, and all but one (a
  reference-“unclear” coded “other”) sit on one of two boundaries.** First, the *nexus* boundary
  (14 cases, mostly reference `d` → predicted `a`): claims alleged on political, ethnic, or
  criminal grounds in which gendered harm nonetheless appears in the facts — a sexual assault in
  police custody inside an imputed-political claim, a threatened rape inside a
  criminal-vengeance claim. The rubric asks whether the *alleged* persecution was gender-based;
  the model tends to code the *presence* of gendered harm, and a pair of mirror-image cases
  shows the boundary wobbles in both directions. Second, the “both” boundary (5 cases): whether
  secondary gender allegations inside a SOGIE claim (rapes, FGM threats against a bisexual
  claimant's family) are substantial enough to count as intersecting (code `c`).
- **At least one reference label is itself questionable.** `566_FC` — a fear of forced
  sterilization under China's family-planning policy — is reference-coded `d` (other), yet the
  rubric expressly lists forced sterilization as gender-based persecution; the model's `a`
  arguably tracks the rubric's letter better than the reference label does. Divergences like
  these are exactly what the human reasonableness review (Appendix 4) is designed to
  adjudicate: on these boundaries, more than one coding can be reasonable.